# Flow-measure regression evaluation

This notebook evaluates the CRNN regression model on the flow-measure validation/test dataset used in the manuscript.

**Expected data layout**

```text
DATA_ROOT/
  angle_level/
    sample_001/
      frame_0001.jpg
      frame_0002.jpg
      ...
    sample_002/
      frame_0001.jpg
      ...
```

The top-level folder name is parsed as `angle_level`, for example `2_8` or `2-8`. The regression target is `[angle, speed]`, where `angle` is taken from the first number in the folder name and `speed` is looked up from the ground-truth Excel file using `(level, angle)`.

**Outputs**

The notebook saves the following files under `OUTPUT_DIR`:

- `predictions.csv`: per-sample true and predicted angle/speed values
- `metrics.json`: MAE, RMSE, bias, standard deviation of error, R², and key evaluation settings
- `metrics.csv`: tabular summary of regression metrics
- `angle_prediction_scatter.png`: true-vs-predicted angle plot
- `angle_residual_histogram.png`: angle residual distribution
- `speed_prediction_scatter.png`: true-vs-predicted flow-speed plot
- `speed_residual_histogram.png`: flow-speed residual distribution
- `evaluation_results.pdf`: combined summary and figures

Update `DATA_ROOT`, `GROUNDTRUTH_XLSX`, and `CHECKPOINT_PATH` in the configuration cell before running.


In [ ]:
# Configuration

from pathlib import Path
import torch

# Update these paths after downloading the validation/test data, ground-truth file, and model checkpoint.
DATA_ROOT = Path("./test")
GROUNDTRUTH_XLSX = Path("./Flow Rate groundtruth.xlsx")
CHECKPOINT_PATH = Path("./checkpoints/best_regressor_val.pth")
OUTPUT_DIR = Path("./results/flow_measure")

MAX_FRAMES = 300
FRAME_STRIDE = 2        # Evaluation uses every second frame, matching the original code.
START_INDEX = 0
BATCH_SIZE = 1          # Keep at 1 unless all samples have the same frame count.
NUM_WORKERS = 0         # Keep at 0 because tensors are moved to DEVICE in the dataset.
SEED = 42
USE_AMP = True          # Automatic mixed precision is used only when CUDA is available.

# Preprocessing settings used for this task.
APPLY_YELLOW_DOT_MASK = True
CENTER_CROP_SIZE = 300
FRAME_SIZE = 224
NORMALIZE_MEAN = 0.5
NORMALIZE_STD = 0.5

# Yellow marker segmentation settings.
YELLOW_HSV_LOWER = (18, 60, 80)
YELLOW_HSV_UPPER = (60, 255, 255)
YELLOW_MIN_AREA = 5
YELLOW_BLUR_KSIZE = 5

# Min-max normalization ranges used during training/evaluation.
ANGLE_MIN_RAW, ANGLE_MAX_RAW = 1.0, 7.0      # Angle index range
SPEED_MIN_RAW, SPEED_MAX_RAW = 9.7, 16.6     # Flow speed range in L/min
ANGLE_INDEX_DEGREE_STEP = 15.0               # angle index 1 -> 0°, index 7 -> 90°

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Using device: {DEVICE}")


In [ ]:
# Imports and reproducibility settings

import json
import math
import random
import re
from contextlib import nullcontext
from glob import glob
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torchvision.models as models
from kornia import augmentation as K
from kornia.augmentation import VideoSequential
from matplotlib.backends.backend_pdf import PdfPages
from torch.utils.data import DataLoader, Dataset

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)


In [ ]:
# Dataset and preprocessing utilities

def yellow_dot_mask(
    img_bgr,
    hsv_lower=YELLOW_HSV_LOWER,
    hsv_upper=YELLOW_HSV_UPPER,
    min_area=YELLOW_MIN_AREA,
    blur_ksize=YELLOW_BLUR_KSIZE,
):
    """Create a binary mask for yellow marker/dot regions in a BGR image."""
    if blur_ksize and blur_ksize > 1:
        img_bgr = cv2.GaussianBlur(img_bgr, (blur_ksize, blur_ksize), 0)

    hsv = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2HSV)

    lower = np.array(hsv_lower, dtype=np.uint8)
    upper = np.array(hsv_upper, dtype=np.uint8)
    mask_hsv = cv2.inRange(hsv, lower, upper)

    v = hsv[..., 2]
    v_eq = cv2.equalizeHist(v)
    _, mask_v = cv2.threshold(v_eq, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)

    mask = cv2.bitwise_and(mask_hsv, mask_v)

    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (3, 3))
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel, iterations=1)
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel, iterations=1)

    num_labels, labels, stats, _ = cv2.connectedComponentsWithStats(mask, connectivity=8)
    clean = np.zeros_like(mask)
    for i in range(1, num_labels):
        if stats[i, cv2.CC_STAT_AREA] >= min_area:
            clean[labels == i] = 255
    return clean


def center_crop(img, target_h, target_w):
    """Center-crop an image, clipping the crop size if the input is smaller."""
    h, w = img.shape[:2]
    crop_h = min(target_h, h)
    crop_w = min(target_w, w)
    y0 = (h - crop_h) // 2
    x0 = (w - crop_w) // 2
    return img[y0 : y0 + crop_h, x0 : x0 + crop_w]


class RegressionDataset(Dataset):
    """Flow-measure regression dataset.

    The top-level folder is parsed as `angle_level`. The first number is the angle index,
    and the second number is the flow level used to look up the speed in the Excel file.
    """

    def __init__(
        self,
        root_dir,
        xlsx_path,
        train=False,
        device=DEVICE,
        max_frames=MAX_FRAMES,
        frame_stride=FRAME_STRIDE,
        start_index=START_INDEX,
        apply_yellow_dot_mask=APPLY_YELLOW_DOT_MASK,
        center_crop_size=CENTER_CROP_SIZE,
        frame_size=FRAME_SIZE,
    ):
        self.root_dir = Path(root_dir)
        self.xlsx_path = Path(xlsx_path)
        self.device = torch.device(device)
        self._train = bool(train)
        self.max_frames = int(max_frames)
        self.frame_stride = int(frame_stride)
        self.start_index = int(start_index)
        self.apply_yellow_dot_mask = bool(apply_yellow_dot_mask)
        self.center_crop_size = int(center_crop_size)
        self.frame_size = int(frame_size)

        if not self.root_dir.exists():
            raise FileNotFoundError(f"DATA_ROOT does not exist: {self.root_dir}")
        if not self.xlsx_path.exists():
            raise FileNotFoundError(f"GROUNDTRUTH_XLSX does not exist: {self.xlsx_path}")

        self.speed_lookup = self._build_speed_lookup(self.xlsx_path)
        self.paths = []       # instance folder paths
        self.targets = []     # raw (angle_index, speed_Lmin) tuples
        self.data = []        # arrays with shape [T, 1, frame_size, frame_size]

        self._set_runtime_transform()
        self._index_folders()
        self._preload_all()

    @property
    def train(self):
        return self._train

    @train.setter
    def train(self, value: bool):
        self._train = bool(value)
        self._set_runtime_transform()

    def _set_runtime_transform(self):
        self.transform = VideoSequential(
            K.Normalize(
                mean=torch.tensor([NORMALIZE_MEAN]),
                std=torch.tensor([NORMALIZE_STD]),
            ),
            same_on_frame=True,
        ).to(self.device)

    @staticmethod
    def _build_speed_lookup(xlsx_path):
        df = pd.read_excel(xlsx_path, engine="openpyxl")
        first_col = df.columns[0]
        df = df.rename(columns={first_col: "Level"}).set_index("Level")

        lookup = {}
        for level in df.index:
            for col in df.columns:
                try:
                    angle = int(str(col).strip())
                except ValueError:
                    continue
                lookup[(int(level), int(angle))] = float(df.loc[level, col])
        return lookup

    def _index_folders(self):
        pattern = re.compile(r"^(\d+)[_-](\d+)$")

        for top_folder in sorted(self.root_dir.iterdir()):
            if not top_folder.is_dir() or top_folder.name.startswith("."):
                continue

            match = pattern.match(top_folder.name)
            if match is None:
                continue

            angle = int(match.group(1))
            level = int(match.group(2))
            key = (level, angle)
            if key not in self.speed_lookup:
                raise ValueError(f"No speed for level={level}, angle={angle} in {self.xlsx_path}.")

            speed = self.speed_lookup[key]
            for instance_folder in sorted(top_folder.iterdir()):
                if not instance_folder.is_dir() or instance_folder.name.startswith("."):
                    continue
                self.paths.append(str(instance_folder))
                self.targets.append((float(angle), float(speed)))

    def _read_and_preprocess_frames(self, folder_path):
        frame_paths = sorted(glob(str(Path(folder_path) / "*.jpg")))
        frames = []

        for frame_path in frame_paths[self.start_index :]:
            img_bgr = cv2.imread(frame_path, cv2.IMREAD_COLOR)
            if img_bgr is None:
                continue

            img_bgr = center_crop(img_bgr, self.center_crop_size, self.center_crop_size)
            img_bgr = cv2.resize(img_bgr, (self.frame_size, self.frame_size), interpolation=cv2.INTER_LINEAR)

            if self.apply_yellow_dot_mask:
                img = yellow_dot_mask(img_bgr)
            else:
                img = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)

            img = (img.astype(np.float32) / 255.0)[None, ...]
            frames.append(img)

            if len(frames) >= self.max_frames:
                break

        if not frames:
            print(f"[WARNING] No valid frames in: {folder_path}")
            return np.zeros((1, 1, self.frame_size, self.frame_size), dtype=np.float32)

        return np.stack(frames, axis=0)

    def _preload_all(self):
        self.data = [self._read_and_preprocess_frames(path) for path in self.paths]

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        frames_np = self.data[idx]
        angle_index, speed_lmin = self.targets[idx]
        folder = self.paths[idx]

        if self._train:
            offset = torch.randint(0, max(self.frame_stride, 1), ()).item()
            frames_np = frames_np[offset :: self.frame_stride]
        else:
            frames_np = frames_np[self.start_index :: self.frame_stride]

        if len(frames_np) == 0:
            frames_np = self.data[idx][:1]

        frames = torch.from_numpy(frames_np).to(self.device, non_blocking=True)
        frames = frames.unsqueeze(0)       # [1, T, 1, H, W]
        frames = self.transform(frames)
        frames = frames.squeeze(0)         # [T, 1, H, W]

        angle_norm = (angle_index - ANGLE_MIN_RAW) / (ANGLE_MAX_RAW - ANGLE_MIN_RAW)
        speed_norm = (speed_lmin - SPEED_MIN_RAW) / (SPEED_MAX_RAW - SPEED_MIN_RAW)
        target = torch.tensor([angle_norm, speed_norm], dtype=torch.float32, device=self.device)

        return frames, target, folder


In [ ]:
# Model definition

class CRNNRegressor(nn.Module):
    """CRNN regressor with a ResNet18 backbone and LSTM temporal aggregation."""

    def __init__(self):
        super().__init__()
        self.resnet = models.resnet18(weights=None)
        self.resnet.conv1 = nn.Conv2d(1, 64, kernel_size=7, stride=2, padding=3, bias=False)
        self.resnet = nn.Sequential(*list(self.resnet.children())[:-1])

        self.lstm = nn.LSTM(input_size=512, hidden_size=128, batch_first=True)
        self.regressor = nn.Sequential(
            nn.Linear(128, 32),
            nn.ReLU(),
            nn.Linear(32, 2),
        )

    def forward(self, x):
        batch_size, timesteps, channels, height, width = x.size()
        x = x.view(batch_size * timesteps, channels, height, width)

        features = self.resnet(x)
        features = features.view(batch_size, timesteps, -1)

        lstm_out, _ = self.lstm(features)
        last_timestep = lstm_out[:, -1, :]
        return self.regressor(last_timestep)


In [ ]:
# Create dataset and dataloader

test_dataset = RegressionDataset(
    root_dir=DATA_ROOT,
    xlsx_path=GROUNDTRUTH_XLSX,
    train=False,
    device=DEVICE,
    max_frames=MAX_FRAMES,
    frame_stride=FRAME_STRIDE,
    start_index=START_INDEX,
    apply_yellow_dot_mask=APPLY_YELLOW_DOT_MASK,
    center_crop_size=CENTER_CROP_SIZE,
    frame_size=FRAME_SIZE,
)

test_dataloader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    drop_last=False,
    num_workers=NUM_WORKERS,
)

target_table = pd.DataFrame(test_dataset.targets, columns=["angle_index", "speed_Lmin"])

print(f"Number of samples: {len(test_dataset)}")
print(f"Number of angle/level combinations: {len(set(Path(p).parent.name for p in test_dataset.paths))}")
print(target_table.describe())


In [ ]:
# Load checkpoint, run evaluation, and save tabular results

def denorm_angle_index(angle_norm):
    return angle_norm * (ANGLE_MAX_RAW - ANGLE_MIN_RAW) + ANGLE_MIN_RAW


def angle_index_to_degrees(angle_index):
    # Maps angle index 1 -> 0°, 7 -> 90°.
    return ANGLE_INDEX_DEGREE_STEP * (angle_index - ANGLE_MIN_RAW)


def denorm_speed(speed_norm):
    return speed_norm * (SPEED_MAX_RAW - SPEED_MIN_RAW) + SPEED_MIN_RAW


def regression_metrics(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    error = y_pred - y_true

    mae = float(np.mean(np.abs(error)))
    rmse = float(math.sqrt(np.mean(error ** 2)))
    bias = float(np.mean(error))
    std_error = float(np.std(error, ddof=1)) if len(error) > 1 else 0.0

    ss_res = float(np.sum((y_true - y_pred) ** 2))
    ss_tot = float(np.sum((y_true - np.mean(y_true)) ** 2))
    r2 = float(1.0 - ss_res / ss_tot) if ss_tot > 0 else None

    return {
        "mae": mae,
        "rmse": rmse,
        "bias": bias,
        "std_error": std_error,
        "r2": r2,
    }


def _strip_module_prefix(state_dict):
    """Remove 'module.' prefixes from DataParallel checkpoints when present."""
    return {
        key.replace("module.", "", 1) if key.startswith("module.") else key: value
        for key, value in state_dict.items()
    }


def _extract_state_dict(checkpoint):
    """Handle common checkpoint formats."""
    if isinstance(checkpoint, nn.Module):
        return checkpoint.state_dict()

    if isinstance(checkpoint, dict):
        for key in ("state_dict", "model_state_dict", "model", "net"):
            value = checkpoint.get(key)
            if isinstance(value, dict):
                return value
            if isinstance(value, nn.Module):
                return value.state_dict()
    return checkpoint


def load_state_dict_safely(model, checkpoint_path, device):
    checkpoint_path = Path(checkpoint_path)
    if not checkpoint_path.exists():
        raise FileNotFoundError(f"Checkpoint file does not exist: {checkpoint_path}")

    try:
        checkpoint = torch.load(checkpoint_path, map_location=device, weights_only=True)
    except Exception:
        # Fallback for older PyTorch versions or checkpoints saved as full model objects.
        checkpoint = torch.load(checkpoint_path, map_location=device)

    state_dict = _extract_state_dict(checkpoint)
    state_dict = _strip_module_prefix(state_dict)
    model.load_state_dict(state_dict)
    return model


@torch.no_grad()
def run_evaluation(dataloader, model, device):
    model.eval()
    rows = []

    if device.type == "cuda" and USE_AMP:
        autocast_context = torch.autocast(device_type="cuda", enabled=True)
    else:
        autocast_context = nullcontext()

    with autocast_context:
        for data, target, folders in dataloader:
            data = data.to(device)
            target = target.to(device)

            pred_norm = model(data)
            if isinstance(pred_norm, (tuple, list)):
                pred_norm = pred_norm[0]

            true_norm = target.detach().cpu().numpy()
            pred_norm_np = pred_norm.detach().cpu().numpy()

            true_angle_index = denorm_angle_index(true_norm[:, 0])
            pred_angle_index = denorm_angle_index(pred_norm_np[:, 0])
            true_angle_deg = angle_index_to_degrees(true_angle_index)
            pred_angle_deg = angle_index_to_degrees(pred_angle_index)

            true_speed = denorm_speed(true_norm[:, 1])
            pred_speed = denorm_speed(pred_norm_np[:, 1])

            for i, folder in enumerate(list(folders)):
                rows.append(
                    {
                        "sample_path": folder,
                        "sample_id": Path(folder).name,
                        "condition_id": Path(folder).parent.name,
                        "angle_true_index": float(true_angle_index[i]),
                        "angle_pred_index": float(pred_angle_index[i]),
                        "angle_true_deg": float(true_angle_deg[i]),
                        "angle_pred_deg": float(pred_angle_deg[i]),
                        "angle_error_deg": float(pred_angle_deg[i] - true_angle_deg[i]),
                        "speed_true_Lmin": float(true_speed[i]),
                        "speed_pred_Lmin": float(pred_speed[i]),
                        "speed_error_Lmin": float(pred_speed[i] - true_speed[i]),
                        "angle_true_norm": float(true_norm[i, 0]),
                        "angle_pred_norm": float(pred_norm_np[i, 0]),
                        "speed_true_norm": float(true_norm[i, 1]),
                        "speed_pred_norm": float(pred_norm_np[i, 1]),
                    }
                )

    return pd.DataFrame(rows)


model = CRNNRegressor()
model = load_state_dict_safely(model, CHECKPOINT_PATH, DEVICE)
model = model.to(DEVICE)

if torch.cuda.device_count() > 1:
    model = nn.DataParallel(model)

prediction_table = run_evaluation(test_dataloader, model, DEVICE)

angle_metrics = regression_metrics(prediction_table["angle_true_deg"], prediction_table["angle_pred_deg"])
speed_metrics = regression_metrics(prediction_table["speed_true_Lmin"], prediction_table["speed_pred_Lmin"])
angle_norm_metrics = regression_metrics(prediction_table["angle_true_norm"], prediction_table["angle_pred_norm"])
speed_norm_metrics = regression_metrics(prediction_table["speed_true_norm"], prediction_table["speed_pred_norm"])

prediction_table.to_csv(OUTPUT_DIR / "predictions.csv", index=False)

metrics_table = pd.DataFrame(
    [
        {"target": "angle", "unit": "degree", **angle_metrics},
        {"target": "flow_speed", "unit": "L/min", **speed_metrics},
        {"target": "angle_normalized", "unit": "normalized", **angle_norm_metrics},
        {"target": "flow_speed_normalized", "unit": "normalized", **speed_norm_metrics},
    ]
)
metrics_table.to_csv(OUTPUT_DIR / "metrics.csv", index=False)

metrics = {
    "task": "flow_measure_regression",
    "num_samples": int(len(prediction_table)),
    "data_root": str(DATA_ROOT),
    "groundtruth_xlsx": str(GROUNDTRUTH_XLSX),
    "checkpoint_path": str(CHECKPOINT_PATH),
    "angle": {"unit": "degree", **angle_metrics},
    "flow_speed": {"unit": "L/min", **speed_metrics},
    "normalized": {
        "angle": angle_norm_metrics,
        "flow_speed": speed_norm_metrics,
        "mean_normalized_mae": float(np.mean([angle_norm_metrics["mae"], speed_norm_metrics["mae"]])),
        "mean_normalized_rmse": float(np.mean([angle_norm_metrics["rmse"], speed_norm_metrics["rmse"]])),
    },
    "preprocessing": {
        "apply_yellow_dot_mask": bool(APPLY_YELLOW_DOT_MASK),
        "yellow_hsv_lower": list(YELLOW_HSV_LOWER),
        "yellow_hsv_upper": list(YELLOW_HSV_UPPER),
        "yellow_min_area": int(YELLOW_MIN_AREA),
        "yellow_blur_ksize": int(YELLOW_BLUR_KSIZE),
        "center_crop_size": int(CENTER_CROP_SIZE),
        "frame_size": int(FRAME_SIZE),
        "max_frames": int(MAX_FRAMES),
        "frame_stride": int(FRAME_STRIDE),
        "normalize_mean": float(NORMALIZE_MEAN),
        "normalize_std": float(NORMALIZE_STD),
        "angle_min_raw": float(ANGLE_MIN_RAW),
        "angle_max_raw": float(ANGLE_MAX_RAW),
        "speed_min_raw": float(SPEED_MIN_RAW),
        "speed_max_raw": float(SPEED_MAX_RAW),
    },
    "model": {
        "architecture": "CRNNRegressor",
        "backbone": "torchvision.models.resnet18(weights=None)",
        "input_channels": 1,
        "lstm_hidden_size": 128,
        "outputs": ["angle_normalized", "flow_speed_normalized"],
    },
}

with open(OUTPUT_DIR / "metrics.json", "w", encoding="utf-8") as f:
    json.dump(metrics, f, indent=2)

print("=== Flow-measure regression metrics ===")
print(metrics_table)
print(f"Saved results to: {OUTPUT_DIR}")


In [ ]:
# Plot and save regression figures

plt.rcParams.update({"font.size": 14})


def _axis_limits(true_values, pred_values):
    all_values = np.concatenate([np.asarray(true_values), np.asarray(pred_values)])
    min_value = float(np.nanmin(all_values))
    max_value = float(np.nanmax(all_values))
    if math.isclose(min_value, max_value):
        min_value -= 1.0
        max_value += 1.0
    padding = 0.05 * (max_value - min_value)
    return min_value - padding, max_value + padding


def save_true_vs_pred_plot(df, true_col, pred_col, title, xlabel, ylabel, output_path):
    fig, ax = plt.subplots(figsize=(7, 6))
    ax.scatter(df[true_col], df[pred_col], s=30, alpha=0.7)
    low, high = _axis_limits(df[true_col], df[pred_col])
    ax.plot([low, high], [low, high], linestyle="--", linewidth=1)
    ax.set_xlim(low, high)
    ax.set_ylim(low, high)
    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)
    ax.set_title(title)
    ax.grid(True, alpha=0.3)
    fig.tight_layout()
    fig.savefig(output_path, dpi=300, bbox_inches="tight")
    plt.show()
    plt.close(fig)


def save_residual_histogram(df, error_col, title, xlabel, output_path):
    fig, ax = plt.subplots(figsize=(7, 6))
    bins = min(30, max(5, int(np.sqrt(len(df)))))
    ax.hist(df[error_col], bins=bins, edgecolor="black")
    ax.axvline(0.0, linestyle="--", linewidth=1)
    ax.set_xlabel(xlabel)
    ax.set_ylabel("Count")
    ax.set_title(title)
    ax.grid(True, alpha=0.3)
    fig.tight_layout()
    fig.savefig(output_path, dpi=300, bbox_inches="tight")
    plt.show()
    plt.close(fig)


angle_scatter_path = OUTPUT_DIR / "angle_prediction_scatter.png"
angle_residual_path = OUTPUT_DIR / "angle_residual_histogram.png"
speed_scatter_path = OUTPUT_DIR / "speed_prediction_scatter.png"
speed_residual_path = OUTPUT_DIR / "speed_residual_histogram.png"

save_true_vs_pred_plot(
    prediction_table,
    "angle_true_deg",
    "angle_pred_deg",
    title=f"Angle prediction (MAE={angle_metrics['mae']:.2f}°, RMSE={angle_metrics['rmse']:.2f}°)",
    xlabel="True angle (°)",
    ylabel="Predicted angle (°)",
    output_path=angle_scatter_path,
)

save_residual_histogram(
    prediction_table,
    "angle_error_deg",
    title="Angle residuals",
    xlabel="Prediction error (°)",
    output_path=angle_residual_path,
)

save_true_vs_pred_plot(
    prediction_table,
    "speed_true_Lmin",
    "speed_pred_Lmin",
    title=f"Flow-speed prediction (MAE={speed_metrics['mae']:.2f} L/min, RMSE={speed_metrics['rmse']:.2f} L/min)",
    xlabel="True flow speed (L/min)",
    ylabel="Predicted flow speed (L/min)",
    output_path=speed_scatter_path,
)

save_residual_histogram(
    prediction_table,
    "speed_error_Lmin",
    title="Flow-speed residuals",
    xlabel="Prediction error (L/min)",
    output_path=speed_residual_path,
)

pdf_path = OUTPUT_DIR / "evaluation_results.pdf"
with PdfPages(pdf_path) as pdf:
    # Summary page
    fig, ax = plt.subplots(figsize=(8, 6))
    ax.axis("off")
    summary_text = (
        "Flow-measure regression evaluation\n\n"
        f"Number of samples: {len(prediction_table)}\n\n"
        f"Angle MAE: {angle_metrics['mae']:.3f}°\n"
        f"Angle RMSE: {angle_metrics['rmse']:.3f}°\n"
        f"Angle R²: {angle_metrics['r2'] if angle_metrics['r2'] is not None else 'NA'}\n\n"
        f"Speed MAE: {speed_metrics['mae']:.3f} L/min\n"
        f"Speed RMSE: {speed_metrics['rmse']:.3f} L/min\n"
        f"Speed R²: {speed_metrics['r2'] if speed_metrics['r2'] is not None else 'NA'}"
    )
    ax.text(0.05, 0.5, summary_text, va="center")
    pdf.savefig(fig, bbox_inches="tight")
    plt.close(fig)

    # Re-create plots for PDF pages.
    for true_col, pred_col, title, xlabel, ylabel in [
        (
            "angle_true_deg",
            "angle_pred_deg",
            f"Angle prediction (MAE={angle_metrics['mae']:.2f}°, RMSE={angle_metrics['rmse']:.2f}°)",
            "True angle (°)",
            "Predicted angle (°)",
        ),
        (
            "speed_true_Lmin",
            "speed_pred_Lmin",
            f"Flow-speed prediction (MAE={speed_metrics['mae']:.2f} L/min, RMSE={speed_metrics['rmse']:.2f} L/min)",
            "True flow speed (L/min)",
            "Predicted flow speed (L/min)",
        ),
    ]:
        fig, ax = plt.subplots(figsize=(7, 6))
        ax.scatter(prediction_table[true_col], prediction_table[pred_col], s=30, alpha=0.7)
        low, high = _axis_limits(prediction_table[true_col], prediction_table[pred_col])
        ax.plot([low, high], [low, high], linestyle="--", linewidth=1)
        ax.set_xlim(low, high)
        ax.set_ylim(low, high)
        ax.set_xlabel(xlabel)
        ax.set_ylabel(ylabel)
        ax.set_title(title)
        ax.grid(True, alpha=0.3)
        fig.tight_layout()
        pdf.savefig(fig, bbox_inches="tight")
        plt.close(fig)

    for error_col, title, xlabel in [
        ("angle_error_deg", "Angle residuals", "Prediction error (°)"),
        ("speed_error_Lmin", "Flow-speed residuals", "Prediction error (L/min)"),
    ]:
        fig, ax = plt.subplots(figsize=(7, 6))
        bins = min(30, max(5, int(np.sqrt(len(prediction_table)))))
        ax.hist(prediction_table[error_col], bins=bins, edgecolor="black")
        ax.axvline(0.0, linestyle="--", linewidth=1)
        ax.set_xlabel(xlabel)
        ax.set_ylabel("Count")
        ax.set_title(title)
        ax.grid(True, alpha=0.3)
        fig.tight_layout()
        pdf.savefig(fig, bbox_inches="tight")
        plt.close(fig)

print(f"Saved figures to: {OUTPUT_DIR}")
print(f"Saved combined PDF to: {pdf_path}")
